In [1]:
from main import run_bot
from demo import run_demo
import config

In [2]:

from config import MIN_R2, OUTPUTS_DIR, TOP_N_HIGHLIGHT, create_run_dirs
from broker.connection import connect_ib
from broker.data import fetch_prices, fetch_prices_free
from broker.orders import calculate_position_size, execute_order, get_portfolio_value
from analysis.universe import fetch_company_metadata, get_sp500_tickers
from analysis.fundamentals import fetch_fundamentals, score_fundamentals, save_fundamentals_csv
from analysis.correlations import compute_correlations, get_top_correlated_pairs, get_top_inverse_pairs
from analysis.model import predict_price
from analysis.signals import generate_signals
from reporting.charts import (plot_correlation_matrix, plot_market_cap_bars,
                               plot_prediction_analysis, plot_price_series)
from reporting.report import print_report, save_signals_csv


In [3]:
n_tickers = None    # int = top N by market cap | None = full S&P 500 | 'FALLBACK_TICKERS' = hardcoded top-20
mode = 'paper'      # demo | paper | live | signals
execute_trades=True
save_plots = True

In [4]:

run_dir, gen_dir, corr_dir = create_run_dirs()

print("\nFetching S&P 500 universe...")
tickers, market_caps = get_sp500_tickers(n=n_tickers)

prices_df = fetch_prices_free(tickers)
if prices_df.empty or len(prices_df.columns) < 5:
    print("✗ Insufficient data. Aborting.")

print("\nCalculating correlations...")
corr_matrix, returns = compute_correlations(prices_df)
top_pairs     = get_top_correlated_pairs(corr_matrix, top_n=10)
inverse_pairs = get_top_inverse_pairs(corr_matrix, top_n=10)

signals_df = generate_signals(prices_df, returns, corr_matrix)

# Enrich signals with company name, sector, founded year, market cap (B)
company_meta = fetch_company_metadata(list(prices_df.columns), market_caps)
signals_df = signals_df.merge(
    company_meta.reset_index().rename(columns={'index': 'ticker'}),
    on='ticker', how='left'
)
# Reorder columns so metadata appears right after ticker
meta_cols = ['company_name', 'sector', 'founded', 'market_cap_B']
other_cols = [c for c in signals_df.columns if c not in ['ticker'] + meta_cols]
signals_df = signals_df[['ticker'] + meta_cols + other_cols]

print_report(signals_df, top_pairs, inverse_pairs)
save_signals_csv(signals_df, run_dir / 'signals.csv')

# Save prices for the dashboard interactive charts
prices_df.to_csv(run_dir / 'prices.csv')

# Fundamental analysis table
fund_raw = fetch_fundamentals(list(prices_df.columns))
fund_df  = score_fundamentals(fund_raw)
save_fundamentals_csv(fund_df, run_dir / 'fundamentals.csv')


  Run outputs → /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50

Fetching S&P 500 universe...
  ✓ 503 tickers fetched from Wikipedia
  Sorting 503 tickers by market cap via yfinance (this takes ~30s)...

Fetching market caps for 503 tickers via yfinance...
  ✓ Market caps retrieved: 503/503
  → Using all 503 S&P 500 tickers



$PPL: possibly delisted; no price data found  (period=1169d)
$CPAY: possibly delisted; no price data found  (period=1169d)
$LNT: possibly delisted; no price data found  (period=1169d)
$MPWR: possibly delisted; no price data found  (period=1169d)
$PODD: possibly delisted; no price data found  (period=1169d)
$COO: possibly delisted; no price data found  (period=1169d)
$WAB: possibly delisted; no price data found  (period=1169d)
$FOX: possibly delisted; no price data found  (period=1169d)
$COR: possibly delisted; no price data found  (period=1169d)
$SPGI: possibly delisted; no price data found  (period=1169d)
$DLTR: possibly delisted; no price data found  (period=1169d)
$BX: possibly delisted; no price data found  (period=1169d)
$CEG: possibly delisted; no price data found  (period=1169d)
$OXY: possibly delisted; no price data found  (period=1169d)
$IDXX: possibly delisted; no price data found  (period=1169d)
$DDOG: possibly delisted; no price data found  (period=1169d)
$PAYX: possibly de

  Tickers with data: 461/503
  ✓ A: 800 bars — last close $114.59
  ✓ AAPL: 800 bars — last close $293.59
  ✓ ABBV: 800 bars — last close $200.94
  ✓ ABNB: 800 bars — last close $142.68
  ✓ ABT: 800 bars — last close $85.75
  ✓ ACGL: 800 bars — last close $93.92
  ✓ ACN: 800 bars — last close $173.91
  ✓ ADBE: 800 bars — last close $248.38
  ✓ ADI: 800 bars — last close $417.11
  ✓ ADM: 800 bars — last close $77.42
  ✓ ADP: 800 bars — last close $209.47
  ✓ ADSK: 800 bars — last close $244.19
  ✓ AEE: 800 bars — last close $108.71
  ✓ AEP: 800 bars — last close $130.93
  ✓ AES: 800 bars — last close $14.27
  ✓ AIG: 800 bars — last close $76.00
  ✓ AIZ: 800 bars — last close $233.31
  ✓ AJG: 800 bars — last close $197.52
  ✓ AKAM: 800 bars — last close $136.73
  ✓ ALB: 800 bars — last close $200.26
  ✓ ALGN: 800 bars — last close $165.36
  ✓ ALL: 800 bars — last close $212.10
  ✓ ALLE: 800 bars — last close $135.85
  ✓ AMAT: 800 bars — last close $430.18
  ✓ AMCR: 800 bars — last close 

In [5]:

# Slice to top N by market cap for a legible heatmap
top_t = [t for t in tickers if t in corr_matrix.columns][:TOP_N_HIGHLIGHT]
plot_correlation_matrix(corr_matrix.loc[top_t, top_t],
                        save_path=corr_dir / 'correlation_matrix.png')

# General/ — price series highlighted by market cap
plot_price_series(prices_df, tickers, top_n=TOP_N_HIGHLIGHT, label='market cap',
                  save_path=gen_dir / 'price_series_market-cap.png')

# General/ — price series highlighted by highest absolute stock price
tickers_by_price = sorted(
    prices_df.columns.tolist(),
    key=lambda t: prices_df[t].iloc[-1],
    reverse=True
)
plot_price_series(prices_df, tickers_by_price, top_n=TOP_N_HIGHLIGHT, label='stock price',
                  save_path=gen_dir / 'price_series_stock-price-absolute.png')

# General/ — price series highlighted by highest normalized return (best performers)
tickers_by_norm = sorted(
    prices_df.columns.tolist(),
    key=lambda t: prices_df[t].iloc[-1] / prices_df[t].iloc[0],
    reverse=True
)
plot_price_series(prices_df, tickers_by_norm, top_n=TOP_N_HIGHLIGHT, label='normalized return',
                  save_path=gen_dir / 'price_series_normalized-return.png')

# General/ — bar chart: top 15 vs bottom 15 by market cap
plot_market_cap_bars(prices_df, tickers, market_caps=market_caps, top_n=TOP_N_HIGHLIGHT,
                     save_path=gen_dir / 'market_cap_bars.png')

# Correlation_method/ — per-ticker prediction analysis
# Generate for: top N by predicted return  +  all BUY/SELL tickers
top_return_tickers = set(signals_df.head(TOP_N_HIGHLIGHT)['ticker'])
buysell_tickers    = set(signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]['ticker'])
analysis_tickers   = top_return_tickers | buysell_tickers

if analysis_tickers:
    print(f"\nGenerating analysis charts ({len(analysis_tickers)} tickers)...")
for ticker in sorted(analysis_tickers):
    pred_ret, r2, top5, corr_signs, y_actual, y_pred = predict_price(
        ticker, returns, corr_matrix
    )
    if y_actual is not None:
        plot_prediction_analysis(
            ticker, returns, prices_df, top5, corr_signs,
            y_actual, y_pred,
            save_path=corr_dir / f'analysis_{ticker}.png'
        )


  Correlation matrix saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/Correlation_method/correlation_matrix.png
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/General/price_series_market-cap.png
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/General/price_series_stock-price-absolute.png
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/General/price_series_normalized-return.png
  Market cap bars saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/General/market_cap_bars.png

Generating analysis charts (18 tickers)...
  Analysis saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/Correlation_method/analysis_CIEN.png
  Analysis saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_15-50/Correlation_method/analysis_COHR.png
  Analysis saved to: /home/patito/Document

In [6]:
execute_trades = 0
if execute_trades:
    ib = connect_ib()
    try:
        portfolio_value = get_portfolio_value(ib)
        print(f"\nPlacing orders (portfolio: ${portfolio_value:,.0f})...")
        actionable = signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]
        for _, row in actionable.iterrows():
            if row['model_r2'] < MIN_R2:
                continue
            strength = min(1.0, row['model_r2'])
            qty = calculate_position_size(portfolio_value, row['current_price'], strength)
            execute_order(ib, row['ticker'], row['signal'], qty)
    finally:
        ib.disconnect()
        print("\n✓ Disconnected from Interactive Brokers.")
else:
    print("\n  ℹ Simulation mode — no orders placed.")
    print("    To execute on paper trading: run_bot(execute_trades=True)")


  ℹ Simulation mode — no orders placed.
    To execute on paper trading: run_bot(execute_trades=True)
